<a href="https://colab.research.google.com/github/PadmaTeja9527/Research-Paper-Question-Answering-System/blob/main/GENAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
pip install pypdf sentence-transformers faiss-cpu transformers torch


In [15]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# -----------------------------
# 1. PDF INGESTION + EXTRACTION
# -----------------------------

pdf_path = "/content/llm" # Changed from '/content/llm.pdf' to '/content/llm'

reader = PdfReader(pdf_path)

chunks = []

for page_no, page in enumerate(reader.pages):

    text = page.extract_text()

    if not text:
        continue

    # -----------------------------
    # 2. CHUNKING
    # chunk size = 800
    # overlap = 150
    # -----------------------------

    for start in range(0, len(text), 650):

        chunk = text[start:start + 800]

        if chunk.strip():
            chunks.append({
                "text": chunk,
                "page": page_no + 1
            })

print("Total chunks:", len(chunks))


# -----------------------------
# 3. CREATE EMBEDDINGS
# -----------------------------

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(texts)

embeddings = np.array(
    embeddings
).astype("float32")


# -----------------------------
# 4. FAISS VECTOR DATABASE
# -----------------------------

index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(embeddings)

print("FAISS index created")


# -----------------------------
# 5. LOAD FLAN-T5 MODEL
# -----------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model_t5 = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

if torch.cuda.is_available():
    model_t5.to("cuda")

print("Language model loaded")


# -----------------------------
# 6. ASK QUESTION
# -----------------------------

question = input(
    "\nAsk a question about the paper: "
)


# -----------------------------
# 7. SEMANTIC RETRIEVAL
# -----------------------------

question_embedding = model.encode(
    [question]
).astype("float32")

distances, ids = index.search(
    question_embedding,
    3
)


# -----------------------------
# 8. BUILD CONTEXT
# -----------------------------

context = ""
sources = []

for i in ids[0]:

    context += (
        f"\nPage {chunks[i]['page']}:\n"
        f"{chunks[i]['text']}\n"
    )

    sources.append(
        chunks[i]["page"]
    )


# -----------------------------
# 9. CONTEXT-GROUNDED GENERATION
# -----------------------------

prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present in the context,
say "Information not found in the paper."

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(model_t5.device)
    for key, value in inputs.items()
}

outputs = model_t5.generate(
    **inputs,
    max_new_tokens=150
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)


# -----------------------------
# 10. DISPLAY ANSWER + CITATION
# -----------------------------

print("\nAnswer:")
print(answer)

print("\nSource pages:")
print(sorted(set(sources)))

Total chunks: 240


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index created


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Language model loaded

Ask a question about the paper: What is the objective of the paper?


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (550 > 512). Running this sequence through the model will result in indexing errors



Answer:
to investigate the following questions: (1) How are LLMs currently applied to NLP tasks in the literature? (2)Have traditional NLP tasks already been solved with LLMs? (3)What is the future of the LLMs for NLP?

Source pages:
[1, 5, 24]
